# Data Cleaning & Enrichment Pipeline
## FAOSTAT Producer Prices — Australia & New Zealand (1991–2025)

**Subject:** Data Visualisation Design and Storytelling — Group Assignment (Parts 2 & 3)  
**Team:** Analysts — Chaitya Nanavati & Shreyas Yadugani  
**Narrative arc:** *The Sparkline* — What Is (current food price volatility) vs What Could Be (stable, food-secure futures)  
**Stakeholder:** UN Food Systems Summit Panel  

---

## Dataset Overview

| # | Source | Role in Story | Join Key | Coverage |
|---|--------|---------------|----------|----------|
| 1 | **FAOSTAT Producer Prices** | Primary dataset | `iso3` + `year` | 1991–2025, AUS & NZL, 120 commodities |
| 2 | **FAO Food Price Index (FFPI)** | Global price benchmark — *the storm* | `year` | 1990–2026, monthly & annual |
| 3 | **Global Hunger Index (GHI)** | Hunger context — *who suffers* | `iso3` | 136 countries, 2000/2008/2016/2025 |
| 4 | **World Bank Food Import %** | Trade vulnerability — *who is exposed* | `iso3` + `year` | 1960–2024, 260+ countries |

---

## How Enrichment Sources Are Used

**FFPI (Enrichment 1):** The FAO Food Price Index provides a globally normalised price benchmark (base 2014–2016 = 100). Joined on `year`, it lets the dashboard show when AUS/NZ commodity prices diverge from or amplify global shocks (2008 food crisis, 2011 Arab Spring spike, 2022 Ukraine war surge). The monthly sheet powers spike annotation overlays on the time-series visual.

**GHI (Enrichment 2):** The Global Hunger Index quantifies hunger severity by country. AUS and NZL are high-income nations not ranked by GHI — their `ghi_2025` is `NaN` by design. The full GHI table is used for the world hunger context choropleth panel. The narrative link: commodities exported from AUS/NZ reach import-dependent, hunger-vulnerable nations — when prices spike, those nations feel it most.

**World Bank (Enrichment 3):** Food imports as % of merchandise imports (indicator `TM.VAL.FOOD.ZS.UN`). Joined on `iso3 + year`, this quantifies which countries are most exposed when AUS/NZ producer prices rise. Powers the what-if scenario slider: *'If wheat prices rise 20%, which nations face the highest compounded risk?'*


---
## 0. Imports & Setup


In [1]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
import os
import warnings
warnings.filterwarnings('ignore')

# Output directory
OUT_DIR = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)

print('Libraries loaded.')
print(f'Output directory: {os.path.abspath(OUT_DIR)}')


Libraries loaded.
Output directory: /content/outputs


---
## 1. FAOSTAT Producer Prices

**File:** `FAOSTAT_data_en_5-1-2026.csv`  
**Encoding:** Latin-1 (FAO standard export — not UTF-8)  
**Countries:** Australia, New Zealand  
**Elements included:** USD/tonne · LCU/tonne · SLC/tonne · Price Index (2014-2016=100)  


### 1.1 Raw load


In [2]:
df_fao = pd.read_csv('FAOSTAT_data_en_5-1-2026.csv',encoding='latin-1',keep_default_na=False)

# Fix BOM character on column 0
df_fao.rename(columns={df_fao.columns[0]: 'Domain Code'}, inplace=True)

print('Raw shape:', df_fao.shape)
print('Columns:', df_fao.columns.tolist())
df_fao.head(3)


Raw shape: (15627, 16)
Columns: ['Domain Code', 'Domain', 'Area Code (M49)', 'Area', 'Element Code', 'Element', 'Item Code (CPC)', 'Item', 'Year Code', 'Year', 'Months Code', 'Months', 'Unit', 'Value', 'Flag', 'Flag Description']


,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Months Code,Months,Unit,Value,Flag,Flag Description
0,PP,Producer Prices,36,Australia,5530,Producer Price (LCU/tonne),01371,"Almonds, in shell",1991,1991,7021,Annual value,LCU,4838.0,A,Official figure
1,PP,Producer Prices,36,Australia,5530,Producer Price (LCU/tonne),01371,"Almonds, in shell",1992,1992,7021,Annual value,LCU,4952.0,A,Official figure
2,PP,Producer Prices,36,Australia,5530,Producer Price (LCU/tonne),01371,"Almonds, in shell",1993,1993,7021,Annual value,LCU,5157.0,A,Official figure


### 1.2 Rename columns & cast types


In [3]:
df_fao.rename(columns={
    'Area':             'country',
    'Item':             'item',
    'Year':             'year',
    'Months':           'month',
    'Value':            'value',
    'Unit':             'unit',
    'Flag':             'flag',
    'Flag Description': 'flag_description',
    'Element':          'element',
}, inplace=True)

df_fao['year']  = pd.to_numeric(df_fao['year'],  errors='coerce').astype('Int64')
df_fao['value'] = pd.to_numeric(df_fao['value'], errors='coerce')

print('Countries:  ', df_fao['country'].unique().tolist())
print('Year range: ', df_fao['year'].min(), '–', df_fao['year'].max())
print('Elements breakdown:')
print(df_fao['element'].value_counts())
print('Flag codes found:')
print(df_fao['flag'].value_counts())


Countries:   ['Australia', 'New Zealand']
Year range:  1991 – 2025
Elements breakdown:
element
Producer Price Index (2014-2016 = 100)    6532
Producer Price (USD/tonne)                3033
Producer Price (SLC/tonne)                3031
Producer Price (LCU/tonne)                3031
Name: count, dtype: int64
Flag codes found:
flag
A    9083
E    6532
X      12
Name: count, dtype: int64


### 1.3 Add ISO3 country codes (primary join key)

FAO uses its own country name strings. We map them to ISO 3166-1 alpha-3 codes,
which are the standard join key for all enrichment sources.


In [4]:
ISO3_MAP = {
    'Australia':   'AUS',
    'New Zealand': 'NZL',
    # Extend this dict if future FAOSTAT downloads include more countries
}

df_fao['iso3'] = df_fao['country'].map(ISO3_MAP)

unmapped = df_fao[df_fao['iso3'].isna()]['country'].unique()
if len(unmapped) == 0:
    print('All countries mapped to ISO3.')
else:
    print(f'Unmapped: {unmapped}')  # must be empty before joining


All countries mapped to ISO3.


### 1.4 Split by element type


In [5]:
# Separate the 4 element types for clarity.
# USD/tonne - cross-country comparison (primary for dashboard)
# Price Index - relative trend analysis (2014-2016 = 100)
# LCU/tonne - raw local-currency value (kept for reference)

df_usd = df_fao[df_fao['element'] == 'Producer Price (USD/tonne)'].copy()
df_idx = df_fao[df_fao['element'] == 'Producer Price Index (2014-2016 = 100)'].copy()
df_lcu = df_fao[df_fao['element'] == 'Producer Price (LCU/tonne)'].copy()

print(f'USD/tonne rows:   {len(df_usd):,}')
print(f'Price Index rows: {len(df_idx):,}')
print(f'LCU/tonne rows:   {len(df_lcu):,}')


USD/tonne rows:   3,033
Price Index rows: 6,532
LCU/tonne rows:   3,031


### 1.5 Handle flags, missing values & zeros

| Flag | Meaning | Action |
|------|---------|--------|
| `A` | Official figure | ✅ Keep |
| `E` | FAO estimate | ✅ Keep — mark `is_imputed = True` |
| `X` | International reliable source | ✅ Keep — mark `is_imputed = True` |
| Zero | Not meaningful as a price | ❌ Drop |
| NaN | No value recorded | ❌ Drop |


In [6]:
def clean_fao(df):
    d = df.copy()
    d['is_imputed'] = d['flag'].isin(['E', 'X'])
    d = d[(d['value'] > 0) & d['value'].notna()].copy() # Drop zero prices and NaN values
    return d

df_usd = clean_fao(df_usd)
df_idx = clean_fao(df_idx)
df_lcu = clean_fao(df_lcu)

print('After cleaning:')
for name, d in [('USD', df_usd), ('IDX', df_idx), ('LCU', df_lcu)]:
    print(f'  {name}: {d.shape}  |  imputed rows: {d["is_imputed"].sum()}')


After cleaning:
  USD: (3033, 18)  |  imputed rows: 4
  IDX: (6532, 18)  |  imputed rows: 6532
  LCU: (3031, 18)  |  imputed rows: 4


### 1.6 Outlier detection — IQR method per (country, commodity) pair

> **Important:** We FLAG outliers but do NOT drop them.  
> The 2008 food crisis, 2011 Arab Spring spike, and 2022 Ukraine war surge are **real events**
> and are central to the Sparkline narrative. Removing them would destroy the story.


In [7]:
def add_outlier_flag(df):
    """Flag values beyond ±3×IQR within each (iso3, item) group."""
    d = df.copy()
    d['is_outlier'] = d.groupby(['iso3', 'item'])['value'].transform(
        lambda x: (
            (x < x.quantile(0.25) - 3 * (x.quantile(0.75) - x.quantile(0.25))) |
            (x > x.quantile(0.75) + 3 * (x.quantile(0.75) - x.quantile(0.25)))
        )
    )
    return d

df_usd = add_outlier_flag(df_usd)
df_idx = add_outlier_flag(df_idx)
df_lcu = add_outlier_flag(df_lcu)

print('Outliers flagged (not dropped):')
for name, d in [('USD', df_usd), ('IDX', df_idx), ('LCU', df_lcu)]:
    print(f'  {name}: {d["is_outlier"].sum()} rows flagged')
print()
print('Sample outlier rows (USD):')
df_usd[df_usd['is_outlier']][['country','item','year','value']].head(8)


Outliers flagged (not dropped):
  USD: 24 rows flagged
  IDX: 21 rows flagged
  LCU: 34 rows flagged

Sample outlier rows (USD):


,country,item,year,value
494,Australia,Asparagus,2011,6897.6
495,Australia,Asparagus,2023,7439.0
496,Australia,Asparagus,2024,7584.5
722,Australia,Bananas,2007,3375.5
1298,Australia,Canary seed,2007,186.6
2364,Australia,Currants,2011,1898.0
2572,Australia,Grapes,2023,2619.7
2573,Australia,Grapes,2024,2657.6


---
## 2. FAO Food Price Index (FFPI) — Enrichment Source 1

**File:** `FAO_Food_Price_Index_.xlsx`  
**Base:** 2014–2016 = 100  
**Sheets used:** `Indices_Monthly_Nominal` (monthly) · `Annual`  

**Narrative use:** Overlaying the FFPI on AUS/NZ producer prices reveals whether local prices
track, lag, or amplify global shocks. The 2022 FFPI reached its highest level since records began.


In [9]:
wb_ffpi = load_workbook('FAO Food Price Index .xlsx', read_only=True)
print('Sheets:', wb_ffpi.sheetnames)


Sheets: ['Indices_Monthly_Nominal', 'Annual', 'Indices_Monthly_Real', 'Annual_Real']


### 2.1 Monthly nominal indices (1990–2026)


In [10]:
FFPI_COLS = ['ffpi_food', 'ffpi_meat', 'ffpi_dairy', 'ffpi_cereals', 'ffpi_oils', 'ffpi_sugar']

rows_m = []
for i, row in enumerate(wb_ffpi['Indices_Monthly_Nominal'].iter_rows(values_only=True)):
    if i < 3 or row[0] is None:
        continue
    rows_m.append({
        'date': row[0],
        **{FFPI_COLS[j]: row[j + 1] for j in range(6)}
    })

df_ffpi_m = pd.DataFrame(rows_m)
df_ffpi_m['date']      = pd.to_datetime(df_ffpi_m['date'])
df_ffpi_m['year']      = df_ffpi_m['date'].dt.year.astype('Int64')
df_ffpi_m['month_num'] = df_ffpi_m['date'].dt.month
for c in FFPI_COLS:
    df_ffpi_m[c] = pd.to_numeric(df_ffpi_m[c], errors='coerce')

print(f'Shape: {df_ffpi_m.shape}')
print(f'Date range: {df_ffpi_m["date"].min().date()} → {df_ffpi_m["date"].max().date()}')
df_ffpi_m.head(4)


Shape: (435, 9)
Date range: 1990-01-01 → 2026-03-01


,date,ffpi_food,ffpi_meat,ffpi_dairy,ffpi_cereals,ffpi_oils,ffpi_sugar,year,month_num
0,1990-01-01,64.444201,74.272219,53.503031,64.140607,44.587672,87.877833,1990,1
1,1990-02-01,64.728299,76.781027,52.218634,62.222377,44.500514,90.662693,1990,2
2,1990-03-01,64.033021,78.543983,41.367123,61.259884,45.745434,95.056585,1990,3
3,1990-04-01,66.015997,81.190035,48.427068,62.820731,44.017094,94.313956,1990,4


### 2.2 Annual indices — join key for FAOSTAT enrichment


In [11]:
rows_a = []
for i, row in enumerate(wb_ffpi['Annual'].iter_rows(values_only=True)):
    if i < 3 or row[0] is None:
        continue
    rows_a.append({
        'year': row[0],
        **{FFPI_COLS[j]: row[j + 1] for j in range(6)}
    })

df_ffpi_annual = pd.DataFrame(rows_a)
df_ffpi_annual['year'] = pd.to_numeric(df_ffpi_annual['year'], errors='coerce').astype('Int64')
for c in FFPI_COLS:
    df_ffpi_annual[c] = pd.to_numeric(df_ffpi_annual[c], errors='coerce')

print(f'Shape: {df_ffpi_annual.shape}')
print(f'Year range: {df_ffpi_annual["year"].min()} – {df_ffpi_annual["year"].max()}')
print()
# Show key crisis years
crisis_years = [2008, 2011, 2022]
print('Key crisis years (FFPI Food Index):')
df_ffpi_annual[df_ffpi_annual['year'].isin(crisis_years)][['year','ffpi_food']]


Shape: (37, 7)
Year range: 1990 – 2026

Key crisis years (FFPI Food Index):


,year,ffpi_food
18,2008,117.737873
21,2011,131.775933
32,2022,144.509866


---
## 3. Global Hunger Index (GHI) — Enrichment Source 2

**File:** `GHI.xlsx` — sheet: *GHI Scores 2025*  
**Coverage:** 136 countries, scores for 2000 · 2008 · 2016 · 2025  
**Scale:** 0 = no hunger, 100 = extremely alarming hunger  

**Narrative use:** Provides the human cost layer. Countries with high GHI scores are often heavily
import-dependent — when AUS/NZ commodity prices rise, these are the nations that suffer most.
AUS and NZL are **not in GHI rankings** (high-income nations) — `ghi_2025 = NaN` is correct.


In [12]:
wb_ghi = load_workbook('GHI.xlsx', read_only=True)
print('Sheets:', wb_ghi.sheetnames)


Sheets: ['GHI Ranking 2025', 'GHI Indicator Values 2025', 'GHI Scores 2025 ']


In [13]:
# Parse GHI Scores sheet
ghi_rows = []
for i, row in enumerate(wb_ghi['GHI Scores 2025 '].iter_rows(values_only=True)):
    if i < 3 or row[0] is None:
        continue
    ghi_rows.append({
        'country_ghi':    row[0],
        'ghi_2000':       row[1],
        'ghi_2008':       row[2],
        'ghi_2016':       row[3],
        'ghi_2025':       row[4],
        'ghi_abs_change': row[5],
        'ghi_pct_change': row[6],
    })

df_ghi = pd.DataFrame(ghi_rows)

# Drop footnote rows
df_ghi = df_ghi[df_ghi['country_ghi'].astype(str).str.len() < 60]

# GHI uses '<5' for very low scores — coerce to NaN (low hunger, not a problem for our story)
def safe_float(x):
    try: return float(x)
    except: return np.nan

for c in ['ghi_2000', 'ghi_2008', 'ghi_2016', 'ghi_2025']:
    df_ghi[c] = df_ghi[c].apply(safe_float)

print(f'GHI records: {df_ghi.shape}')
df_ghi.head(5)


GHI records: (136, 7)


,country_ghi,ghi_2000,ghi_2008,ghi_2016,ghi_2025,ghi_abs_change,ghi_pct_change
0,Afghanistan,49.6,32.7,28.0,29.0,1,3.4
1,Albania,15.3,15.3,6.7,7.0,0.3,4.3
2,Algeria,14.1,10.8,8.0,7.1,-0.9,-12.7
3,Angola,63.8,35.3,25.7,29.7,4,13.5
4,Argentina,6.5,5.2,5.3,6.4,1.1,17.2


### 3.1 Map GHI country names → ISO3


In [16]:
GHI_ISO3 = {
    'Afghanistan':'AFG', 'Albania':'ALB', 'Algeria':'DZA', 'Angola':'AGO',
    'Argentina':'ARG', 'Armenia':'ARM', 'Azerbaijan':'AZE', 'Bangladesh':'BGD',
    'Belarus':'BLR', 'Bolivia (Plurinat. State of)':'BOL',
    'Bosnia & Herzegovina':'BIH', 'Botswana':'BWA', 'Brazil':'BRA',
    'Bulgaria':'BGR', 'Burkina Faso':'BFA', 'Burundi':'BDI',
    'Cabo Verde':'CPV', 'Cambodia':'KHM', 'Cameroon':'CMR',
    'Central African Republic':'CAF', 'Chad':'TCD', 'Chile':'CHL',
    'China':'CHN', 'Colombia':'COL', 'Comoros':'COM',
    'Congo (Republic of)':'COG', 'Costa Rica':'CRI', 'Croatia':'HRV',
    "Côte d'Ivoire":'CIV', 'Dem. Rep. of the Congo':'COD',
    'Djibouti':'DJI', 'Dominican Republic':'DOM', 'Ecuador':'ECU',
    'Egypt':'EGY', 'El Salvador':'SLV', 'Equatorial Guinea':'GNQ',
    'Eritrea':'ERI', 'Estonia':'EST', 'Eswatini':'SWZ', 'Ethiopia':'ETH',
    'Fiji':'FJI', 'Gabon':'GAB', 'Gambia':'GMB', 'Georgia':'GEO',
    'Ghana':'GHA', 'Guatemala':'GTM', 'Guinea':'GIN', 'Guinea-Bissau':'GNB',
    'Guyana':'GUY', 'Haiti':'HTI', 'Honduras':'HND', 'Hungary':'HUN',
    'India':'IND', 'Indonesia':'IDN', 'Iran (Islamic Republic of)':'IRN',
    'Iraq':'IRQ', 'Jamaica':'JAM', 'Jordan':'JOR', 'Kazakhstan':'KAZ',
    'Kenya':'KEN', 'Korea (DPR)':'PRK', 'Kuwait':'KWT', 'Kyrgyzstan':'KGZ',
    'Lao PDR':'LAO', 'Latvia':'LVA', 'Lebanon':'LBN', 'Lesotho':'LSO',
    'Liberia':'LBR', 'Libya':'LBY', 'Lithuania':'LTU', 'Madagascar':'MDG',
    'Malawi':'MWI', 'Maldives':'MDV', 'Malaysia':'MYS', 'Mali':'MLI',
    'Mauritania':'MRT', 'Mauritius':'MUS', 'Mexico':'MEX',
    'Moldova (Rep. of)':'MDA', 'Mongolia':'MNG', 'Montenegro':'MNE',
    'Morocco':'MAR', 'Mozambique':'MOZ', 'Myanmar':'MMR', 'Namibia':'NAM',
    'Nepal':'NPL', 'Nicaragua':'NIC', 'Niger':'NER', 'Nigeria':'NGA',
    'North Macedonia':'MKD', 'Oman':'OMN', 'Pakistan':'PAK', 'Panama':'PAN',
    'Papua New Guinea':'PNG', 'Paraguay':'PRY', 'Peru':'PER',
    'Philippines':'PHL', 'Qatar':'QAT', 'Romania':'ROU',
    'Russian Federation':'RUS', 'Rwanda':'RWA', 'Saudi Arabia':'SAU',
    'Senegal':'SEN', 'Serbia':'SRB', 'Sierra Leone':'SLE', 'Slovakia':'SVK',
    'Solomon Islands':'SLB', 'Somalia':'SOM', 'South Africa':'ZAF',
    'South Sudan':'SSD', 'Sri Lanka':'LKA', 'Sudan':'SDN', 'Suriname':'SUR',
    'Syrian Arab Republic':'SYR', 'Tajikistan':'TJK',
    'Tanzania (United Rep. of)':'TZA', 'Thailand':'THA', 'Timor-Leste':'TLS',
    'Togo':'TGO', 'Trinidad & Tobago':'TTO', 'Tunisia':'TUN',
    'Turkmenistan':'TKM', 'Türkiye':'TUR', 'Uganda':'UGA', 'Ukraine':'UKR',
    'United Arab Emirates':'ARE', 'Uruguay':'URY', 'Uzbekistan':'UZB',
    'Venezuela (Boliv. Rep. of)':'VEN', 'Viet Nam':'VNM',
    'Yemen':'YEM', 'Zambia':'ZMB', 'Zimbabwe':'ZWE',
    'Bahrain':'BHR', 'Benin':'BEN', 'Bhutan':'BTN',
    'Australia':'AUS', 'New Zealand':'NZL',
}

df_ghi['iso3'] = df_ghi['country_ghi'].map(GHI_ISO3)

n_mapped   = df_ghi['iso3'].notna().sum()
n_unmapped = df_ghi['iso3'].isna().sum()
print(f'Mapped:   {n_mapped}/{len(df_ghi)}')
print(f'Unmapped: {n_unmapped} - {df_ghi[df_ghi["iso3"].isna()]["country_ghi"].tolist()}')


Mapped:   136/136
Unmapped: 0 - []


### 3.2 Top 10 most food-insecure countries (GHI 2025)
These are the nations most likely to be harmed when global food prices spike.


In [17]:
top_hungry = (df_ghi[df_ghi['ghi_2025'].notna()]
    .nlargest(10, 'ghi_2025')[['country_ghi','iso3','ghi_2025','ghi_2016','ghi_2000']]
    .reset_index(drop=True)
)
top_hungry


,country_ghi,iso3,ghi_2025,ghi_2016,ghi_2000
0,Somalia,SOM,42.6,49.4,64.3
1,Dem. Rep. of the Congo,COD,37.5,36.4,46.1
2,South Sudan,SSD,37.5,NaN,NaN
3,Madagascar,MDG,35.8,35.0,42.0
4,Haiti,HTI,35.7,29.9,40.2
5,Chad,TCD,34.8,38.5,49.6
6,Niger,NER,33.9,33.3,52.7
7,Central African Republic,CAF,33.4,36.0,46.8
8,Nigeria,NGA,32.8,29.9,38.2
9,Papua New Guinea,PNG,31.0,31.9,31.3


---
## 4. World Bank Food Import Dependency — Enrichment Source 3

**File:** `Wroldbank_dataset.csv`  
**Indicator:** `TM.VAL.FOOD.ZS.UN` — Food imports as % of merchandise imports  
**Format:** Wide (countries × year columns) → melted to long

**Narrative use:** High food import dependency + high commodity prices = vulnerability.
This powers the what-if scenario: *'If AUS/NZ wheat prices rise X%, these countries' food
import bills increase by Y% — and they already spend Z% of export earnings on food.'*


In [19]:
df_wb_raw = pd.read_csv('Wroldbank dataset.csv', skiprows=4, encoding='utf-8-sig')
print('Shape (wide):', df_wb_raw.shape)
print('Columns sample:', df_wb_raw.columns[:6].tolist())


Shape (wide): (266, 71)
Columns sample: ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961']


In [20]:
# Identify year columns
year_cols = [c for c in df_wb_raw.columns if str(c).strip().isdigit()]
print(f'Year columns found: {len(year_cols)} ({year_cols[0]} – {year_cols[-1]})')

df_wb = df_wb_raw[['Country Name', 'Country Code'] + year_cols].melt(
    id_vars=['Country Name', 'Country Code'],
    var_name='year',
    value_name='food_import_pct'
)
df_wb.rename(columns={'Country Name': 'country_wb','Country Code': 'iso3',}, inplace=True)

df_wb['year']            = pd.to_numeric(df_wb['year'], errors='coerce').astype('Int64')
df_wb['food_import_pct'] = pd.to_numeric(df_wb['food_import_pct'], errors='coerce')
df_wb = df_wb.dropna(subset=['food_import_pct'])

print(f'Long format shape: {df_wb.shape}')
print(f'Countries: {df_wb["iso3"].nunique()}')
print(f'Year range: {df_wb["year"].min()} – {df_wb["year"].max()}')
df_wb.head(4)


Year columns found: 66 (1960 – 2025)
Long format shape: (9984, 4)
Countries: 246
Year range: 1962 – 2024


,country_wb,iso3,year,food_import_pct
534,Afghanistan,AFG,1962,7.835267
535,Africa Western and Central,AFW,1962,18.032962
536,Angola,AGO,1962,19.341476
541,Argentina,ARG,1962,3.893525


In [21]:
# AUS and NZL food import dependency over time
print('Australia food import % (last 10 years):')
print(df_wb[df_wb['iso3']=='AUS'].sort_values('year').tail(10)[['year','food_import_pct']].to_string(index=False))
print()
print('New Zealand food import % (last 10 years):')
print(df_wb[df_wb['iso3']=='NZL'].sort_values('year').tail(10)[['year','food_import_pct']].to_string(index=False))


Australia food import % (last 10 years):
 year  food_import_pct
 2015         6.562559
 2016         7.100550
 2017         6.482228
 2018         6.525947
 2019         7.164694
 2020         7.597665
 2021         6.444211
 2022         6.149459
 2023         6.122724
 2024         6.447674

New Zealand food import % (last 10 years):
 year  food_import_pct
 2015        11.363120
 2016        11.441339
 2017        11.526239
 2018        11.325567
 2019        11.525817
 2020        13.414119
 2021        11.475669
 2022        11.067179
 2023        11.268232
 2024        12.473764


### 4.1 Top 15 most food-import-dependent countries (latest year available)


In [22]:
latest_wb = ( df_wb.sort_values('year', ascending=False)
    .groupby('iso3')
    .first()
    .reset_index()
    .nlargest(15, 'food_import_pct')
    [['iso3', 'country_wb', 'year', 'food_import_pct']]
    .reset_index(drop=True)
)
latest_wb


,iso3,country_wb,year,food_import_pct
0,ERI,Eritrea,2003,45.571379
1,COM,Comoros,2023,42.067988
2,GNB,Guinea-Bissau,2018,41.710352
3,KIR,Kiribati,2021,41.444406
4,YEM,"Yemen, Rep.",2019,39.091366
5,DJI,Djibouti,2023,38.561234
6,NER,Niger,2024,37.894364
7,HTI,Haiti,2016,37.261399
8,BEN,Benin,2024,33.269959
9,CPV,Cabo Verde,2024,32.560888


---
## 5. Enrich: Join All Sources onto FAOSTAT

Three left joins — left joins preserve ALL FAOSTAT rows even if enrichment data is missing:

```
FAOSTAT (USD/tonne)                         - base table
   LEFT JOIN  FFPI Annual    ON year        - global price benchmark
   LEFT JOIN  World Bank     ON iso3+year   - food import dependency
   LEFT JOIN  GHI 2025       ON iso3        - hunger severity snapshot
```


In [23]:
# Prepare slim versions for joining
ghi_slim = df_ghi[['iso3', 'ghi_2025']].dropna(subset=['iso3'])
wb_slim  = df_wb[['iso3', 'year', 'food_import_pct']]

def enrich(df):
    """Apply all three enrichment joins to a FAOSTAT dataframe."""
    return (
        df
        .merge(df_ffpi_annual, on='year',           how='left')  # Enrichment 1: FFPI
        .merge(wb_slim,        on=['iso3', 'year'], how='left')  # Enrichment 3: World Bank
        .merge(ghi_slim,       on='iso3',           how='left')  # Enrichment 2: GHI
    )

master_usd = enrich(df_usd)
master_idx = enrich(df_idx)

print(f'Master USD enriched: {master_usd.shape}')
print(f'Master IDX enriched: {master_idx.shape}')
print()
print('Null check on joined columns:')
null_report = master_usd[['food_import_pct', 'ghi_2025', 'ffpi_food']].isnull().sum()
print(null_report)
print()
print('Note: ghi_2025 is NaN for AUS/NZL — expected (high-income nations not in GHI ranking)')


Master USD enriched: (3033, 27)
Master IDX enriched: (6532, 27)

Null check on joined columns:
food_import_pct       0
ghi_2025           3033
ffpi_food             0
dtype: int64

Note: ghi_2025 is NaN for AUS/NZL — expected (high-income nations not in GHI ranking)


In [24]:
# Preview the fully enriched master dataset
master_usd[['country', 'iso3', 'item', 'year', 'value','ffpi_food', 'food_import_pct', 'ghi_2025','is_imputed', 'is_outlier'
]].head(10)


,country,iso3,item,year,value,ffpi_food,food_import_pct,ghi_2025,is_imputed,is_outlier
0,Australia,AUS,"Almonds, in shell",1991,3768.6,62.345284,5.219889,NaN,False,False
1,Australia,AUS,"Almonds, in shell",1992,3636.8,64.225559,5.043264,NaN,False,False
2,Australia,AUS,"Almonds, in shell",1993,3506.8,62.259405,4.987079,NaN,False,False
3,Australia,AUS,"Almonds, in shell",1994,4730.4,67.260547,4.999334,NaN,False,False
4,Australia,AUS,"Almonds, in shell",1995,4203.0,76.828785,4.956548,NaN,False,False
5,Australia,AUS,"Almonds, in shell",1996,6229.1,77.820863,4.852902,NaN,False,False
6,Australia,AUS,"Almonds, in shell",1997,4928.1,70.756824,4.910177,NaN,False,False
7,Australia,AUS,"Almonds, in shell",1998,3951.4,64.823765,4.737756,NaN,False,False
8,Australia,AUS,"Almonds, in shell",1999,3335.6,55.384947,4.804384,NaN,False,False
9,Australia,AUS,"Almonds, in shell",2000,2632.1,53.658254,4.554207,NaN,False,False


---
## 6. Data Quality Checks


In [25]:
print('MASTER PRODUCER PRICES (USD/tonne)')
print(f'  Rows:          {len(master_usd):,}')
print(f'  Countries:     {master_usd["country"].unique().tolist()}')
print(f'  Commodities:   {master_usd["item"].nunique()} unique items')
print(f'  Year range:    {master_usd["year"].min()} – {master_usd["year"].max()}')
print(f'  Outlier rows:  {master_usd["is_outlier"].sum()}')
print(f'  Imputed rows:  {master_usd["is_imputed"].sum()}')
print()
print('Value summary (USD/tonne):')
print(master_usd['value'].describe().round(2))


MASTER PRODUCER PRICES (USD/tonne)
  Rows:          3,033
  Countries:     ['Australia', 'New Zealand']
  Commodities:   96 unique items
  Year range:    1991 – 2024
  Outlier rows:  24
  Imputed rows:  4

Value summary (USD/tonne):
count     3033.00
mean      1300.25
std       1851.84
min         11.90
25%        280.80
50%        699.10
75%       1618.70
max      36133.40
Name: value, dtype: float64


In [27]:
# FFPI coverage — are all FAOSTAT years covered?
fao_years  = set(master_usd['year'].dropna().astype(int))
ffpi_years = set(df_ffpi_annual['year'].dropna().astype(int))
missing = sorted(fao_years - ffpi_years)
print(f'FAOSTAT years not in FFPI: {missing if missing else "None — full coverage"}')


FAOSTAT years not in FFPI: None — full coverage


In [28]:
# Top 10 most expensive commodities on average
print('Top 10 highest average producer prices (USD/tonne):')
(
    master_usd.groupby('item')['value']
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
    .rename(columns={'value': 'avg_usd_per_tonne'})
    .assign(avg_usd_per_tonne=lambda x: x['avg_usd_per_tonne'].round(2))
)


Top 10 highest average producer prices (USD/tonne):


,item,avg_usd_per_tonne
0,Raspberries,8718.20
1,"Pistachios, in shell",7346.55
2,Blueberries,6454.10
3,Cherries,5599.53
4,Hop cones,4778.88
5,"Almonds, in shell",4479.53
6,Unmanufactured tobacco,4265.01
7,Green garlic,4165.50
8,Strawberries,3801.05
9,Mushrooms and truffles,3549.22


In [29]:
# Price trend check — AUS wheat (a key global export)
wheat = master_usd[
    (master_usd['country']=='Australia') &
    (master_usd['item'].str.contains('Wheat', case=False))
].sort_values('year')[['year','value','ffpi_food','food_import_pct']]

if len(wheat) > 0:
    print('Australia Wheat prices vs FFPI (sample years):')
    print(wheat.iloc[::5].to_string(index=False))  # every 5th row
else:
    print('Note: Wheat not in this FAOSTAT slice — check item list.')
    print('Available items sample:', master_usd['item'].unique()[:10].tolist())


Australia Wheat prices vs FFPI (sample years):
 year  value  ffpi_food  food_import_pct
 1991  102.8  62.345284         5.219889
 1996  203.5  77.820863         4.852902
 2001  120.0  55.353759         4.942220
 2006  152.9  72.851982         4.770725
 2011  265.1 131.775933         5.142196
 2016  205.9  91.953037         7.100550
 2021  231.8 125.734070         6.444211


In [30]:
# Outlier flagged for dashboard
outliers = master_usd[master_usd['is_outlier']][['country','item','year','value']]
print(f'Flagged outliers ({len(outliers)} rows):')
outliers.sort_values('value', ascending=False)


Flagged outliers (24 rows):


,country,item,year,value
639,Australia,Hop cones,2009,14701.4
640,Australia,Hop cones,2011,10583.2
118,Australia,Asparagus,2024,7584.5
117,Australia,Asparagus,2023,7439.0
116,Australia,Asparagus,2011,6897.6
1821,Australia,Spinach,2014,6747.9
2073,Australia,"Walnuts, in shell",2011,6106.5
160,Australia,Bananas,2007,3375.5
1427,Australia,Persimmons,2024,3366.5
1426,Australia,Persimmons,2023,3279.9


---
## 7. Save All Output Files

| File | Description | Primary use in dashboard |
|------|-------------|-------------------------|
| `master_producer_prices_usd.csv` | Primary enriched dataset | All 4 visuals |
| `master_producer_price_index.csv` | Price index (2014-2016=100) + enrichments | Trend / sparkline visual |
| `master_producer_prices_lcu.csv` | LCU/tonne (local currency, cleaned) | Reference only |
| `ffpi_monthly.csv` | FFPI monthly 1990–2026 | Spike annotation timeline |
| `ffpi_annual.csv` | FFPI annual 1990–2025 | Year-on-year comparison |
| `ghi_cleaned.csv` | GHI scores 136 countries | Hunger context choropleth |
| `worldbank_food_import_pct.csv` | Food import % all countries | What-if scenario slider |


In [31]:
file_map = {
    'master_producer_prices_usd.csv':  master_usd,
    'master_producer_price_index.csv': master_idx,
    'master_producer_prices_lcu.csv':  df_lcu,
    'ffpi_monthly.csv':                df_ffpi_m,
    'ffpi_annual.csv':                 df_ffpi_annual,
    'ghi_cleaned.csv':                 df_ghi,
    'worldbank_food_import_pct.csv':   df_wb,
}

for fname, df in file_map.items():
    path = f'{OUT_DIR}/{fname}'
    df.to_csv(path, index=False)

---
## 8. Data Dictionary

### `master_producer_prices_usd.csv` — primary enriched file (27 columns)

| Column | Type | Source | Description |
|--------|------|--------|-------------|
| `country` | string | FAOSTAT | FAO country name |
| `iso3` | string | Derived | ISO 3166-1 alpha-3 code (join key) |
| `item` | string | FAOSTAT | Commodity name (e.g. 'Wheat', 'Apples') |
| `year` | Int64 | FAOSTAT | Observation year (1991–2025) |
| `month` | string | FAOSTAT | 'Annual value' for yearly obs; month name for monthly |
| `value` | float | FAOSTAT | Producer price in USD per tonne (cleaned: >0, non-null) |
| `unit` | string | FAOSTAT | 'USD' |
| `flag` | string | FAOSTAT | Data quality: A=official, E=estimated, X=intl sources |
| `is_imputed` | bool | Derived | True if flag ∈ {E, X} |
| `is_outlier` | bool | Derived | True if value > Q3+3×IQR or < Q1−3×IQR within (iso3, item) |
| `ffpi_food` | float | FAO FFPI | FAO Food Price Index (2014-2016=100), annual, global |
| `ffpi_meat` | float | FAO FFPI | Meat Price Index |
| `ffpi_dairy` | float | FAO FFPI | Dairy Price Index |
| `ffpi_cereals` | float | FAO FFPI | Cereals Price Index |
| `ffpi_oils` | float | FAO FFPI | Vegetable Oils Price Index |
| `ffpi_sugar` | float | FAO FFPI | Sugar Price Index |
| `food_import_pct` | float | World Bank | Food imports as % of merchandise imports |
| `ghi_2025` | float | GHI | Hunger Index score 2025 (0–100); NaN for AUS/NZL |

---

### Join logic
```
FAOSTAT USD  LEFT JOIN  FFPI Annual    ON  year
             LEFT JOIN  World Bank     ON  iso3 + year
             LEFT JOIN  GHI 2025       ON  iso3
```
---
*Pipeline authored by: Chaitya Nanavati & Shreyas Yadugani (Analysts)*  